# 形態素解析とサブワード分割（分割単位・正規化・未知語処理）

書籍 第3章「系列ラベリング問題の解き方」（3-2 形態素解析：単語分割の基準・サブワード化）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、以下の内容を体験します。

1. `SudachiPy` による形態素解析と分割単位 (Mode A / B / C) の違い、品詞・見出し語化・正規化形（3-2「単語分割の基準」）
2. 未知語・新語に対する辞書ベース形態素解析 (`SudachiPy`) の振る舞いと `SentencePiece` によるサブワード分割の比較（3-2「サブワード化」）

## 1. 準備

形態素解析器 `SudachiPy`（辞書 `sudachidict_core`）およびサブワード分割器 `SentencePiece` をインストールします。

In [1]:
!pip install -q sudachipy sudachidict_core sentencepiece

In [2]:
import tempfile
from pathlib import Path

import pandas as pd
import sentencepiece as spm
from IPython.display import display
from sudachipy import dictionary, tokenizer

---

## 2. 形態素解析と分割単位（3-2「単語分割の基準」）

日本語は英語と異なり単語の境界が空白で区切られていないため、タスクや目的に応じて「どこまでを 1 単語とするか（分割単位）」を決める必要があります。
`SudachiPy` では、1 つの辞書で以下の 3 つの分割モードを切り替えられます。

- **Mode A（短単位相当）**: 最も細かい分割（検索や詳細な言語学的分析向け）
- **Mode B（中単位相当）**: 複合語の一部をまとめた中間的な分割
- **Mode C（長単位相当）**: 固有名詞や複合語をひとまとまりとして扱う分割（固有表現認識や情報抽出向け）

In [3]:
sudachi = dictionary.Dictionary().create()

sample_texts = [
    "東京都立大学で自然言語処理を学ぶ。",
    "関西国際空港から新型コロナウイルス感染症の対策会議へ向かう。",
]

mode_map = {
    "Mode A（短単位）": tokenizer.Tokenizer.SplitMode.A,
    "Mode B（中単位）": tokenizer.Tokenizer.SplitMode.B,
    "Mode C（長単位）": tokenizer.Tokenizer.SplitMode.C,
}

for text in sample_texts:
    print(f"入力文: {text}")
    for name, mode in mode_map.items():
        tokens = [m.surface() for m in sudachi.tokenize(text, mode)]
        print(f"  {name}: {' / '.join(tokens)}")
    print()

入力文: 東京都立大学で自然言語処理を学ぶ。
  Mode A（短単位）: 東京 / 都立 / 大学 / で / 自然 / 言語 / 処理 / を / 学ぶ / 。
  Mode B（中単位）: 東京 / 都立 / 大学 / で / 自然 / 言語 / 処理 / を / 学ぶ / 。
  Mode C（長単位）: 東京都立大学 / で / 自然言語処理 / を / 学ぶ / 。

入力文: 関西国際空港から新型コロナウイルス感染症の対策会議へ向かう。
  Mode A（短単位）: 関西 / 国際 / 空港 / から / 新型 / コロナ / ウイルス / 感染 / 症 / の / 対策 / 会議 / へ / 向かう / 。
  Mode B（中単位）: 関西 / 国際 / 空港 / から / 新型 / コロナ / ウイルス / 感染症 / の / 対策 / 会議 / へ / 向かう / 。
  Mode C（長単位）: 関西国際空港 / から / 新型コロナウイルス感染症 / の / 対策 / 会議 / へ / 向かう / 。



### 品詞情報・見出し語化（正規化）の確認

形態素解析では単語分割だけでなく、品詞（POS）や辞書形（終止形）・正規化形（表記揺れの統一）も同時に得られます。

In [4]:
text = "走ったり泳いだりして、コーヒーを飲みました。"
rows = []
for m in sudachi.tokenize(text, tokenizer.Tokenizer.SplitMode.C):
    rows.append({
        "表層形": m.surface(),
        "辞書形（見出し語）": m.dictionary_form(),
        "正規化形": m.normalized_form(),
        "品詞（大分類-中分類）": "-".join(m.part_of_speech()[:2]),
        "読み": m.reading_form(),
    })
pd.DataFrame(rows)

,表層形,辞書形（見出し語）,正規化形,品詞（大分類-中分類）,読み
0,走っ,走る,走る,動詞-一般,ハシッ
1,たり,たり,たり,助詞-副助詞,タリ
2,泳い,泳ぐ,泳ぐ,動詞-一般,オヨイ
3,だり,だり,たり,助詞-副助詞,ダリ
4,し,する,為る,動詞-非自立可能,シ
5,て,て,て,助詞-接続助詞,テ
6,、,、,、,補助記号-読点,、
7,コーヒー,コーヒー,コーヒー,名詞-普通名詞,コーヒー
8,を,を,を,助詞-格助詞,ヲ
9,飲み,飲む,飲む,動詞-一般,ノミ


---

## 3. 未知語・新語に対する辞書ベース形態素解析とサブワード分割の比較（3-2「サブワード化」）

辞書ベースの形態素解析では、辞書に載っていない新語・固有名詞が未知語 (`OOV`) として文字種単位でまとめて切り出されたり、複合語が既知語の細かな断片に分割されたりすることがあります。
一方、ニューラルモデル（BERT や機械翻訳など）では、コーパスから統計的に部分語（サブワード）の語彙を学習する **SentencePiece**（BPE やユニグラム言語モデル）がよく使われます。

小さなコーパスから `SentencePiece` モデルを学習し、辞書ベースの `SudachiPy` (Mode C) と `SentencePiece` で、学習データに出現する文と未知語・新語を含む文がそれぞれどのように分割されるかを比較してみましょう。

In [5]:
mini_corpus = [
    "自然言語処理の教科書を読みました。",
    "形態素解析は単語分割と品詞推定を行います。",
    "固有表現認識では人名や組織名を抽出します。",
    "東京都立大学の研究室で自然言語処理を研究しています。",
    "機械翻訳と文書要約は系列変換タスクです。",
    "深層学習と事前学習モデルが広く使われています。",
] * 15  # 頻度確保のため複製

with tempfile.TemporaryDirectory() as tmpdir:
    corpus_path = Path(tmpdir) / "corpus.txt"
    model_prefix = str(Path(tmpdir) / "sp_model")
    corpus_path.write_text("\n".join(mini_corpus), encoding="utf-8")

    spm.SentencePieceTrainer.train(
        input=str(corpus_path),
        model_prefix=model_prefix,
        vocab_size=360,
        model_type="unigram",
        character_coverage=1.0,
        byte_fallback=True,
        hard_vocab_limit=False,
        minloglevel=2,
        bos_id=-1,
        eos_id=-1,
    )
    sp = spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")

    test_sentences = [
        "自然言語処理の研究をしています。",
        "ChatGPTで未知語を解析する。",
        "HuggingFaceでLLMをファインチューニングする。",
    ]

    for s in test_sentences:
        sudachi_tokens = [
            f"{m.surface()}[OOV]" if m.is_oov() else m.surface()
            for m in sudachi.tokenize(s, tokenizer.Tokenizer.SplitMode.C)
        ]
        sp_pieces = sp.encode_as_pieces(s)
        print("入力:", s)
        print("  SudachiPy (Mode C):", " / ".join(sudachi_tokens))
        print("  SentencePiece     :", sp_pieces)

入力: 自然言語処理の研究をしています。
  SudachiPy (Mode C): 自然言語処理 / の / 研究 / を / し / て / い / ます / 。
  SentencePiece     : ['▁', '自', '然', '言', '語', '処', '理', 'の', '研究', 'を', 'し', 'て', 'い', 'ま', 'す', '。']
入力: ChatGPTで未知語を解析する。
  SudachiPy (Mode C): ChatGPT / で / 未知語 / を / 解析 / する / 。
  SentencePiece     : ['▁', '<0x43>', '<0x68>', '<0x61>', '<0x74>', '<0x47>', '<0x50>', '<0x54>', 'で', '<0xE6>', '<0x9C>', '<0xAA>', '<0xE7>', '<0x9F>', '<0xA5>', '語', 'を', '解', '析', 'す', '<0xE3>', '<0x82>', '<0x8B>', '。']
入力: HuggingFaceでLLMをファインチューニングする。
  SudachiPy (Mode C): HuggingFace[OOV] / で / LLM / を / ファイン / チューニング / する / 。
  SentencePiece     : ['▁', '<0x48>', '<0x75>', '<0x67>', '<0x67>', '<0x69>', '<0x6E>', '<0x67>', '<0x46>', '<0x61>', '<0x63>', '<0x65>', 'で', '<0x4C>', '<0x4C>', '<0x4D>', 'を', '<0xE3>', '<0x83>', '<0x95>', '<0xE3>', '<0x82>', '<0xA1>', '<0xE3>', '<0x82>', '<0xA4>', '<0xE3>', '<0x83>', '<0xB3>', '<0xE3>', '<0x83>', '<0x81>', '<0xE3>', '<0x83>', '<0xA5>', '<0xE3>', '<0x83>', '<0xBC>', '<0xE3>',

- **`SudachiPy`（辞書ベース）**: 辞書に登録されている複合語（「自然言語処理」など）や最新辞書に収録済みの語（`ChatGPT` など）は 1 語として認識されますが、未登録の新語（`HuggingFace` など）は `[OOV]`（未知語）として文字種ルールで切り出されたり、`ファインチューニング` のように既知の構成要素へ分割されたりします（ドメイン固有名詞を辞書に追加する方法は [形態素解析のユーザ辞書作成と単体テスト](ch5_dictionary.ipynb) を参照してください）。
- **`SentencePiece`（サブワード分割）**: 学習コーパスにない単語であっても、未知語トークン `<unk>` 1 つに潰れるのではなく、文字単位や既知の部分列（`byte_fallback=True` により必要に応じてバイト列）へフォールバックして表現できます（`▁` は単語の先頭または空白を表す記号です）。

### 次の演習への案内

- **3-2 後半（単語ラティス・Viterbi アルゴリズムと CRF vs MEMM）**: [単語ラティス・Viterbi アルゴリズムと CRF vs MEMM](ch3_viterbi_crf.ipynb)
- **3-1 固有表現認識（旧第 4〜5 節の BIO タグ・`sklearn-crfsuite` による素性テンプレートと CRF 実験を移設統合）**: [固有表現認識の基礎（ガゼッティア・BIO タグ・素性テンプレートと CRF・サブワード整合）](ch3_ner_gazetteer_subword.ipynb)